# ReceiptGuard-ML Kaggle Training Notebook

This notebook runs the complete ReceiptGuard-ML pipeline on Kaggle.
Each cell imports from existing pipeline modules - no logic duplication.

## Cell 1: Setup & Install Dependencies

In [ ]:
# Install required packages for Kaggle environment
!pip install -q torch torchvision transformers accelerate
!pip install -q Pillow numpy scikit-learn pandas tqdm matplotlib seaborn
!pip install -q tensorboard python-dotenv pytesseract

# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

## Cell 2: Mount SROIE2019 Dataset

In [ ]:
## Cell 2: Mount SROIE2019 Dataset

import os
import sys
from pathlib import Path

# 1. Setup Base Paths
KAGGLE_INPUT_PATH = Path('/kaggle/input')
WORKING_PATH = Path('/kaggle/working')

# 2. Set the precise path found via console
# Structure: /kaggle/input/datasets/urbikn/sroie-datasetv2/SROIE2019
RAW_DATA_PATH = KAGGLE_INPUT_PATH / 'datasets' / 'urbikn' / 'sroie-datasetv2' / 'SROIE2019'

print(f"--- 📂 Dataset Initialization ---")

# 3. Verification & Content Listing
if RAW_DATA_PATH.exists():
    print(f"✅ Dataset Path Verified: {RAW_DATA_PATH}")
    
    # Identify the core folders inside SROIE2019
    print("\n📊 Detected Components:")
    for item in RAW_DATA_PATH.iterdir():
        if item.is_dir():
            # Get a count of files inside to ensure it's loaded correctly
            files = list(item.glob('*'))
            print(f"  • {item.name}/ -> {len(files)} items found")
        else:
            print(f"  • {item.name}")
else:
    print("❌ ERROR: Path not found. Double-check the folder names.")
    # Fallback search if the username 'urbikn' changes
    print("Searching for any 'SROIE2019' folder...")
    fallback = list(KAGGLE_INPUT_PATH.rglob('SROIE2019'))
    if fallback:
        print(f"Found alternative path: {fallback[0]}")
        RAW_DATA_PATH = fallback[0]

# 4. Integrate Project Code (ReceiptGuard) - Enhanced search
print(f"\n--- 🔍 Searching for Receipt_Guard ---")

# First try exact path
RECEIPTGUARD_PATH = KAGGLE_INPUT_PATH / 'Receipt_Guard'

# If not found, search recursively
if not RECEIPTGUARD_PATH.exists():
    print(f"❌ Receipt_Guard not found at {RECEIPTGUARD_PATH}")
    print("🔍 Searching recursively for Receipt_Guard...")
    rg_matches = list(KAGGLE_INPUT_PATH.rglob('Receipt_Guard'))
    
    if rg_matches:
        # Filter for directories, not files
        rg_dirs = [m for m in rg_matches if m.is_dir()]
        if rg_dirs:
            RECEIPTGUARD_PATH = rg_dirs[0]
            print(f"✅ Found Receipt_Guard at: {RECEIPTGUARD_PATH}")
        else:
            # If only files found, use parent directory
            RECEIPTGUARD_PATH = rg_matches[0].parent
            print(f"✅ Found Receipt_Guard (file) at: {rg_matches[0]}")
            print(f"   Using parent directory: {RECEIPTGUARD_PATH}")
    else:
        print("❌ ERROR: Receipt_Guard not found anywhere in /kaggle/input")
        print("📂 Available directories in /kaggle/input:")
        if KAGGLE_INPUT_PATH.exists():
            for item in KAGGLE_INPUT_PATH.iterdir():
                if item.is_dir():
                    print(f"  • {item.name}/")
        raise FileNotFoundError("Receipt_Guard not found. Please upload the project to Kaggle.")

# Now add to sys.path
if RECEIPTGUARD_PATH.exists():
    sys.path.insert(0, str(RECEIPTGUARD_PATH))
    sys.path.insert(0, str(RECEIPTGUARD_PATH / 'src'))
    print(f"\n✅ Project 'Receipt_Guard' added to system path.")
    print(f"  Project path: {RECEIPTGUARD_PATH}")
    print(f"  SRC path: {RECEIPTGUARD_PATH / 'src'}")
    
    # Verify config.py exists
    config_check = RECEIPTGUARD_PATH / 'config.py'
    config_src_check = RECEIPTGUARD_PATH / 'src' / 'config.py'
    
    print(f"\n📋 Config file verification:")
    print(f"  Root config.py: {'✅' if config_check.exists() else '❌'} {config_check}")
    print(f"  src/config.py: {'✅' if config_src_check.exists() else '❌'} {config_src_check}")
else:
    raise FileNotFoundError(f"RECEIPTGUARD_PATH does not exist: {RECEIPTGUARD_PATH}")

# 5. Verify sys.path
print(f"\n🐍 Python Path Check (first 10 entries):")
for i, path in enumerate(sys.path[:10]):
    print(f"  {i}: {path}")

## Cell 3: Run Preprocessing Pipeline

In [ ]:
## Cell 3: Run Preprocessing Pipeline

import sys
from pathlib import Path

# Ensure project paths are in sys.path (self-contained, doesn't rely on Cell 2)
KAGGLE_INPUT_PATH = Path('/kaggle/input')
WORKING_PATH = Path('/kaggle/working')

# Find and add Receipt_Guard to sys.path
print("🔍 Searching for Receipt_Guard...")
rg_found = False
rg_path = None

# Search locations: input, working, and working/Receipt_Guard (from git clone)
search_paths = [
    KAGGLE_INPUT_PATH,
    WORKING_PATH,
    WORKING_PATH / 'Receipt_Guard',
]

for search_base in search_paths:
    print(f"  Checking: {search_base}")
    
    # Try exact path
    exact_path = search_base / 'Receipt_Guard'
    if exact_path.exists():
        rg_path = exact_path
        rg_found = True
        print(f"✅ Found at exact path: {rg_path}")
        break
    
    # Search recursively within this base
    if search_base.exists():
        rg_matches = list(search_base.rglob('Receipt_Guard'))
        rg_dirs = [m for m in rg_matches if m.is_dir()]
        if rg_dirs:
            rg_path = rg_dirs[0]
            rg_found = True
            print(f"✅ Found via search: {rg_path}")
            break

if not rg_found:
    print("❌ ERROR: Receipt_Guard not found")
    print("📂 Searched in:")
    for p in search_paths:
        exists = "✅" if p.exists() else "❌"
        print(f"  {exists} {p}")
    
    print("\n📂 Available directories in /kaggle/working:")
    if WORKING_PATH.exists():
        for item in WORKING_PATH.iterdir():
            if item.is_dir():
                print(f"  • {item.name}/")
    
    print("\n📂 Available directories in /kaggle/input:")
    if KAGGLE_INPUT_PATH.exists():
        for item in KAGGLE_INPUT_PATH.iterdir():
            if item.is_dir():
                print(f"  • {item.name}/")
    
    raise FileNotFoundError(
        "Receipt_Guard not found. Please clone the repo or upload as a dataset:\n"
        "  !git clone https://github.com/MoeenUddin01/Receipt_Guard.git"
    )

# Add paths to sys.path
sys.path.insert(0, str(rg_path))
sys.path.insert(0, str(rg_path / 'src'))
print(f"\n✅ Added to sys.path:")
print(f"  {rg_path}")
print(f"  {rg_path / 'src'}")

# Debug: Check current Python path
print(f"\n🐍 Python Path Check (first 10 entries):")
for i, path in enumerate(sys.path[:10]):
    print(f"  {i}: {path}")

# Debug: Check what's available
print(f"\n📂 Contents of project root ({rg_path}):")
for item in sorted(rg_path.iterdir())[:15]:  # Limit to first 15 items
    if item.is_file():
        print(f"  📄 {item.name}")
    elif item.is_dir():
        print(f"  📁 {item.name}/")

print(f"\n📂 Contents of src directory ({rg_path / 'src'}):")
src_dir = rg_path / 'src'
if src_dir.exists():
    for item in sorted(src_dir.iterdir())[:15]:
        if item.is_file():
            print(f"  📄 {item.name}")
        elif item.is_dir():
            print(f"  📁 {item.name}/")
else:
    print(f"  ⚠️  src directory not found")

# Try importing config - handle both root and src locations
print("\n🔍 Attempting config import...")
CFG = None
override_config = None

try:
    from config import CFG, override_config
    print(f"✅ Imported config from root (config.py)")
except ImportError as e:
    print(f"❌ Failed to import config from root: {e}")
    try:
        from src.config import CFG, override_config
        print(f"✅ Imported config from src.config")
    except ImportError as e2:
        print(f"❌ Failed to import config from src.config: {e2}")
        print(f"\n⚠️  config.py not found. Please ensure config.py exists in the project root.")
        raise ImportError("Cannot import config from either root or src/config.py")

# Try importing preprocessing_pipeline
print("\n🔍 Attempting preprocessing_pipeline import...")
try:
    from pipelines.preprocessing_pipeline import (
        run_preprocessing_pipeline,
        PreprocessingConfig
    )
    print(f"✅ Imported preprocessing_pipeline from pipelines")
except ImportError as e:
    print(f"❌ Failed to import from pipelines: {e}")
    try:
        from src.pipelines.preprocessing_pipeline import (
            run_preprocessing_pipeline,
            PreprocessingConfig
        )
        print(f"✅ Imported preprocessing_pipeline from src.pipelines")
    except ImportError as e2:
        print(f"❌ Failed to import from src.pipelines: {e2}")
        raise ImportError("Cannot import preprocessing_pipeline")

print("\n" + "="*50)
print("🚀 Starting Preprocessing Pipeline")
print("="*50)

override_config({
    'paths.raw_data_dir': str(RAW_DATA_PATH),
    'paths.artifacts_dir': str(WORKING_PATH / 'artifacts'),
    'kaggle.input_path': str(KAGGLE_INPUT_PATH),
    'kaggle.working_path': str(WORKING_PATH),
    'training.batch_size': 16,  # Kaggle GPU optimized
    'training.num_epochs': 15,
})

print(f"✅ Configuration overridden for Kaggle environment")
print(f"  Raw data: {CFG.paths.raw_data_dir}")
print(f"  Artifacts: {CFG.paths.artifacts_dir}")

# Define preprocessing configuration using CFG
preprocess_config = PreprocessingConfig(
    raw_data_path=CFG.paths.raw_data_dir,
    processed_data_path=str(WORKING_PATH / 'processed'),
    splits=['train', 'test'],
    verify_images=True
)

print("Starting preprocessing...")
preprocess_summary = run_preprocessing_pipeline(preprocess_config)

print(f"\nPreprocessing complete!")
print(f"Total samples: {preprocess_summary.get('total_samples', 0)}")
print(f"Failed samples: {preprocess_summary.get('failed_samples_count', 0)}")
print(f"Output directory: {preprocess_config.processed_data_path}")

## Cell 4: Training with Kaggle-Optimized Config (batch_size=16, GPU)

In [ ]:
## Cell 4: Training with Kaggle-Optimized Config (batch_size=16, GPU)

from pipelines.model_training_pipeline import (
    run_training_pipeline,
    TrainingConfig
)

# GPU optimization check
import torch
if torch.cuda.is_available():
    print(f"🚀 GPU detected: {torch.cuda.get_device_name(0)}")
    override_config({'training.batch_size': 16})  # Larger batch for GPU
else:
    print("⚠️  CPU detected - using smaller batch size")
    override_config({'training.batch_size': 8})

# Kaggle-optimized training configuration using CFG
training_config = TrainingConfig(
    model_path=str(RAW_DATA_PATH / 'layoutlm-base-uncased'),
    num_labels=9,  # O, B-COMPANY, I-COMPANY, B-DATE, I-DATE, B-ADDRESS, I-ADDRESS, B-TOTAL, I-TOTAL
    dropout=CFG.model.dropout,
    output_dir=str(WORKING_PATH / 'checkpoints'),
    num_epochs=CFG.training.num_epochs,
    batch_size=CFG.training.batch_size,
    max_length=CFG.data.max_length,
    learning_rate=CFG.training.learning_rate,
    weight_decay=CFG.training.weight_decay,
    warmup_ratio=CFG.training.warmup_ratio,
    seed=CFG.training.seed,
    data_path=str(RAW_DATA_PATH)  # FIXED: Use Kaggle path, not CFG default
)

print("Starting training with Kaggle-optimized config...")
print(f"Batch size: {training_config.batch_size}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Data path: {training_config.data_path}")

# Run training
training_summary = run_training_pipeline(training_config)

# Print results
if training_summary.get('status') == 'completed':
    print(f"\nTraining completed successfully!")
    print(f"Best eval loss: {training_summary.get('best_eval_loss', 'N/A'):.4f}")
    print(f"Best checkpoint: {training_summary.get('best_checkpoint', 'N/A')}")
    print(f"Total epochs: {training_summary.get('epochs_trained', 'N/A')}")
else:
    print(f"Training failed: {training_summary.get('error', 'Unknown error')}")

## Cell 5: Evaluation Pipeline

In [ ]:
## Cell 5: Evaluation Pipeline

from pipelines.evaluation_pipeline import (
    run_evaluation_pipeline,
    EvaluationConfig
)

# Find the best checkpoint
checkpoint_path = WORKING_PATH / 'checkpoints' / 'best_model.pt'
if not checkpoint_path.exists():
    # Find any .pt file
    pt_files = list((WORKING_PATH / 'checkpoints').glob('*.pt'))
    if pt_files:
        checkpoint_path = pt_files[0]
        print(f"Using checkpoint: {checkpoint_path}")
    else:
        raise FileNotFoundError("No checkpoint found!")
else:
    print(f"Using best checkpoint: {checkpoint_path}")

# Define evaluation configuration using CFG
eval_config = EvaluationConfig(
    checkpoint_path=str(checkpoint_path),
    model_path=CFG.model.model_path,
    processed_data_path=str(WORKING_PATH / 'processed'),
    output_dir=CFG.paths.evaluation_dir,
    batch_size=CFG.inference.batch_size,
    run_fraud_detection=True
)

print("\nStarting evaluation...")
eval_summary = run_evaluation_pipeline(eval_config)

# Print results
ner_metrics = eval_summary.get('ner_metrics', {})
macro_f1 = ner_metrics.get('macro', {}).get('f1', 0.0)

print(f"\nEvaluation Results:")
print(f"Macro F1 Score: {macro_f1:.4f}")

# Per-entity scores
print("\nPer-Entity F1 Scores:")
for entity, metrics in ner_metrics.get('per_entity', {}).items():
    print(f"  {entity}: {metrics['f1']:.4f}")

# Fraud detection results
fraud_report = eval_summary.get('fraud_report', {})
if fraud_report.get('status') == 'completed':
    print(f"\nFraud Detection:")
    print(f"  Total processed: {fraud_report.get('total_processed', 0)}")
    print(f"  Duplicates found: {fraud_report.get('duplicates_found', 0)}")
    print(f"  Duplicate rate: {fraud_report.get('duplicate_rate', 0):.2%}")

## Cell 6: Save Output to /kaggle/working/

In [ ]:
import json
from datetime import datetime

# Create final summary report
final_report = {
    'timestamp': datetime.now().isoformat(),
    'kaggle_config': {
        'batch_size': 16,
        'num_epochs': 15,
        'device': str(torch.cuda.get_device_name(0)) if torch.cuda.is_available() else 'CPU',
        'gpu_count': torch.cuda.device_count()
    },
    'preprocessing': {
        'total_samples': preprocess_summary.get('total_samples', 0),
        'failed_samples': preprocess_summary.get('failed_samples_count', 0)
    },
    'training': {
        'status': training_summary.get('status'),
        'best_eval_loss': training_summary.get('best_eval_loss'),
        'best_checkpoint': training_summary.get('best_checkpoint'),
        'epochs_trained': training_summary.get('epochs_trained'),
        'training_time_seconds': training_summary.get('training_time_seconds')
    },
    'evaluation': {
        'macro_f1': ner_metrics.get('macro', {}).get('f1'),
        'per_entity_f1': {
            entity: metrics['f1']
            for entity, metrics in ner_metrics.get('per_entity', {}).items()
        },
        'duplicates_detected': fraud_report.get('duplicates_found', 0)
    },
    'output_paths': {
        'processed_data': str(WORKING_PATH / 'processed'),
        'checkpoints': str(WORKING_PATH / 'checkpoints'),
        'evaluation': str(WORKING_PATH / 'evaluation'),
        'best_model': str(checkpoint_path)
    }
}

# Save final report
report_path = WORKING_PATH / 'final_report.json'
with open(report_path, 'w') as f:
    json.dump(final_report, f, indent=2)
print(f"Final report saved to: {report_path}")

# Print summary of saved outputs
print("\n" + "="*60)
print("OUTPUT SUMMARY - All files saved to /kaggle/working/")
print("="*60)

for item in WORKING_PATH.rglob('*'):
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        rel_path = item.relative_to(WORKING_PATH)
        print(f"  {rel_path} ({size_mb:.2f} MB)")

print("="*60)
print(f"\nBest model checkpoint: {checkpoint_path}")
print(f"Evaluation results: {WORKING_PATH / 'evaluation'}")
print(f"Final report: {report_path}")
print("\nAll outputs are preserved in /kaggle/working/ for download!")